# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and associated `@id`s for reference when extracting and analyzing data.

In [ ]:
# List all record sets and their fields by `@id`
print("Available record sets with fields:")
record_sets = []
for rset in metadata.record_sets:
    print(f"- RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    field_ids = [fld.id for fld in rset.fields]
    field_names = [fld.name for fld in rset.fields]
    print(f"  Fields: {list(zip(field_names, field_ids))}")
    record_sets.append(rset.id)
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the relevant record set and field `@id`s identified above.

In [ ]:
# Load each record set as a DataFrame by `@id`
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2), "\n")

# Example: Use the first record set for further analysis
target_record_set_id = record_sets[0]
# List first few rows
dataframes[target_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing a numeric field, and grouping by a category. All fields are referenced by their `@id`.

In [ ]:
# --- Choose field IDs for this dataset (look at code in section 2 above) ---
df = dataframes[target_record_set_id]
print(f"Columns in selected record set ({target_record_set_id}):", df.columns.tolist())

# For demonstration, we try to find a plausible numeric field and a grouping (categorical) field by typical mass spec dataset columns
sample_numeric_field = None
sample_group_field = None
for col in df.columns:
    if ('abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'peptide' in col.lower() or 'coverage' in col.lower()) and df[col].dtype in [float, int]:
        sample_numeric_field = col
        break
if not sample_numeric_field:
    # Fallback: try first float/int column
    for col in df.columns:
        if df[col].dtype in [float, int]:
            sample_numeric_field = col
            break
# Try to find a grouping field
for col in df.columns:
    if ('sample' in col.lower() or 'group' in col.lower() or 'category' in col.lower()) and df[col].dtype == object:
        sample_group_field = col
        break
if not sample_group_field:
    # Try a common protein identifier if present
    for col in df.columns:
        if ('protein' in col.lower() or 'accession' in col.lower() or 'id' == col.lower()) and df[col].dtype == object:
            sample_group_field = col
            break

print(f"Using numeric field (as @id): {sample_numeric_field}")
print(f"Using grouping field: {sample_group_field}")

# --- Filtering, normalizing, and grouping ---
if sample_numeric_field:
    # Drop missing
    filtered_df = df.dropna(subset=[sample_numeric_field]).copy()
    # Set threshold as e.g. mean for illustration
    threshold = filtered_df[sample_numeric_field].mean()
    filtered_df = filtered_df[filtered_df[sample_numeric_field] > threshold].copy()
    print(f"Filtered records with {sample_numeric_field} > mean ({threshold:.2f}):")
    print(filtered_df.head())
    # Normalization
    norm_col = f"{sample_numeric_field}_normalized"
    filtered_df[norm_col] = (
        filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()
    ) / (filtered_df[sample_numeric_field].std() + 1e-9)
    print(f"\nNormalized {sample_numeric_field} for filtered records:")
    print(filtered_df[[sample_numeric_field, norm_col]].head())
    if sample_group_field and sample_group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(sample_group_field)[sample_numeric_field].mean().reset_index()
        )
        print(f"\nGrouped data by {sample_group_field} (mean {sample_numeric_field}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and group-wise averages if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[sample_numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {sample_numeric_field}")
    plt.xlabel(sample_numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if sample_group_field and sample_group_field in df.columns:
        groupstats = df.groupby(sample_group_field)[sample_numeric_field].mean().sort_values(ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(
            x=groupstats.index.astype(str), y=groupstats.values
        )
        plt.title(f"Mean {sample_numeric_field} by {sample_group_field}")
        plt.xlabel(sample_group_field)
        plt.ylabel(f"Mean {sample_numeric_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("Skipping visualization: No numeric field found.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to programmatically load and explore the FAIR^2 mass spectrometry dataset for human extracellular vesicle proteins. We extracted and summarized the available record sets and fields using their Croissant `@id`, performed basic EDA including filtering and normalization of a numeric field, examined group-wise patterns, and visualized field distributions. 

MLCommons Croissant enables reliable, standards-based programmatic data access and transparency. Explore the dataset further by modifying thresholds, visualizing other fields, or integrating with downstream ML pipelines.